## ⚖️ 05. Operating Point (최적 임계값) 산출 방안

AI 모델이 내놓은 모델의 Confidence(0~1)를 바탕으로, 실제 병변을 감지했다고 판정할 기준선(Threshold)을 결정합니다.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, confusion_matrix
import os

print('Loading real test predictions...')
# 실제 파일 기반
try:
    df = pd.read_csv('../checkpoints/test_predictions.csv')
except FileNotFoundError:
    print('⚠️ test_predictions.csv 가 checkpoints 에 없습니다. 평가 스크립트를 먼저 실행하세요.')
    df = pd.DataFrame()

if not df.empty and 'Cardiomegaly_prob' in df.columns and 'Cardiomegaly_true' in df.columns:
    y_true = df['Cardiomegaly_true'].values
    y_prob = df['Cardiomegaly_prob'].values
    
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    
    # Youden's J
    j_idx = np.argmax(tpr - fpr)
    j_thresh = thresholds[j_idx]
    
    # calc metric for a given threshold
    def calc_metrics(t):
        tn, fp, fn, tp = confusion_matrix(y_true, y_prob >= t).ravel()
        sens = tp / (tp + fn) if (tp+fn)>0 else 0
        spec = tn / (tn + fp) if (tn+fp)>0 else 0
        ppv = tp / (tp + fp) if (tp+fp)>0 else 0
        npv = tn / (tn + fn) if (tn+fn)>0 else 0
        return t, sens, spec, ppv, npv

    metrics = [
        ("Youden's J", *calc_metrics(j_thresh)),
        ("Sensitivity 90%", *calc_metrics(np.percentile(thresholds[tpr >= 0.9], 10) if any(tpr >= 0.9) else j_thresh)),
        ("Specificity 90%", *calc_metrics(np.percentile(thresholds[1-fpr >= 0.9], 10) if any(1-fpr >= 0.9) else j_thresh))
    ]
    
    op_df = pd.DataFrame([
        {'기준': m[0], 'Threshold': m[1], 'Sensitivity': m[2], 'Specificity': m[3], 'PPV': m[4], 'NPV': m[5]}
        for m in metrics
    ])
    
    print('=== Operating Points (Cardiomegaly) ===')
    display(op_df)
    
    # 대시보드를 위해 저장
    op_df.to_csv('../checkpoints/op_analysis.csv', index=False)
    print('✅ Saved to checkpoints/op_analysis.csv')
